# Scope 3.1 Carbon Data Catalog
**Portfolioprojekt | Datenmanagement | Juni 2026**

Prototypischer Datenkatalog zur systematischen Identifikation, Dokumentation und
Lückenerkennung von Scope-3.1-Emissionsdaten auf Basis des GHG Protocol.
Vollständige Projektbeschreibung: `01_kontext.md`

## Semantische Beziehungskette

Requirement → Business Term → Technical Field → Dataset → Source System
                                                    ↓
                                               Data Owner
                                                    ↓
                                            Metadata Gap

Eine Anforderung gilt als vollständig abgedeckt, wenn die gesamte Kette
lückenlos dokumentiert und einem Owner zugewiesen ist.

## Aufbau dieses Notebooks

| Abschnitt | Inhalt |
|---|---|
| **1** | Datenbankschema & Initialisierung |
| **2** | Seed-Daten: 8 Scope-3.1-Anforderungen |
| **3** | Gap Detection Engine |
| **4** | Interaktive Abfrage & Curation |
| **5** | Coverage-Visualisierung |
| **6** | Fazit & Ausblick |

In [1]:
#Umgebungscheck
import sqlite3
import pandas as pd

try:
    import plotly.express as px
    print("✓ plotly verfügbar")
except ImportError:
    print("→ pip install plotly")

try:
    import ipywidgets as widgets
    print("✓ ipywidgets verfügbar")
except ImportError:
    print("→ pip install ipywidgets")

print("✓ sqlite3 und pandas bereit")

→ pip install plotly
→ pip install ipywidgets
✓ sqlite3 und pandas bereit


In [2]:
import sqlite3
import pandas as pd

# Datenbankverbindung herstellen
conn = sqlite3.connect("catalog.db")
cursor = conn.cursor()

# Fremdschlüssel aktivieren
cursor.execute("PRAGMA foreign_keys = ON")

print("✓ catalog.db erstellt und verbunden")

✓ catalog.db erstellt und verbunden


In [3]:
import sqlite3
import pandas as pd

# ── Datenbankverbindung ──────────────────────────────────────────
conn = sqlite3.connect("catalog.db")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON")

# ── Tabellen in korrekter Reihenfolge erstellen ──────────────────
# Reihenfolge ist kritisch: referenzierte Tabelle muss zuerst existieren

# 1. Quellsysteme
cursor.execute("""
CREATE TABLE IF NOT EXISTS source_systems (
    id                      INTEGER PRIMARY KEY AUTOINCREMENT,
    name                    TEXT NOT NULL,
    system_type             TEXT NOT NULL,  -- ERP | Buchhaltung | CRM | Konsolidierung | informell
    subsidiary              TEXT NOT NULL,  -- z.B. "FloorTec MY"
    country                 TEXT NOT NULL,
    currency                TEXT NOT NULL,
    is_central              INTEGER NOT NULL DEFAULT 0,  -- 0=Nein, 1=Ja
    consolidation_status    TEXT NOT NULL,  -- vollständig | partiell | nicht_angeschlossen
    consolidation_target    TEXT,           -- Zielsystem z.B. "SAP BW EU", sonst NULL
    data_availability       TEXT NOT NULL,  -- vollständig | partiell | unbekannt
    owner                   TEXT,           -- NULL = Gap
    notes                   TEXT
)
""")

# 2. Datensätze
cursor.execute("""
CREATE TABLE IF NOT EXISTS datasets (
    id                          INTEGER PRIMARY KEY AUTOINCREMENT,
    name                        TEXT NOT NULL,
    description                 TEXT,
    source_system_id            INTEGER NOT NULL,
    data_contact                TEXT,           -- NULL = Gap
    data_contact_role           TEXT,           -- Controlling | IT | GL-Assistenz | unbekannt
    data_contact_reliability    TEXT,           -- hoch | mittel | niedrig
    currency                    TEXT,
    exchange_rate_date          TEXT,           -- NULL = Gap bei Fremdwährung
    data_availability           TEXT NOT NULL,  -- vollständig | partiell | unbekannt
    FOREIGN KEY (source_system_id) REFERENCES source_systems(id)
)
""")

# 3. Technische Felder
cursor.execute("""
CREATE TABLE IF NOT EXISTS technical_fields (
    id                              INTEGER PRIMARY KEY AUTOINCREMENT,
    dataset_id                      INTEGER NOT NULL,
    field_name                      TEXT NOT NULL,  -- z.B. LIFNR, MENGE
    field_description               TEXT,
    data_type                       TEXT,           -- TEXT | NUMERIC | DATE
    unit                            TEXT,           -- kg | EUR | pcs | unbekannt
    unit_system                     TEXT,           -- metrisch | imperial | lokal | gemischt
    material_description_quality    TEXT,           -- lesbar | teilweise_lesbar | unlesbar | fehlend
    is_nullable                     INTEGER DEFAULT 1,  -- 0=Pflichtfeld, 1=optional
    FOREIGN KEY (dataset_id) REFERENCES datasets(id)
)
""")

# 4. Business Terms
cursor.execute("""
CREATE TABLE IF NOT EXISTS business_terms (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,
    name            TEXT NOT NULL,
    definition      TEXT,           -- NULL = Gap
    owner           TEXT,           -- NULL = Gap
    domain          TEXT,           -- Einkauf | Finanzen | Nachhaltigkeit
    tier_level      INTEGER         -- 1 | 2 | 3 nach GHG Protocol
)
""")

# 5. Anforderungen
cursor.execute("""
CREATE TABLE IF NOT EXISTS requirements (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    name                TEXT NOT NULL,
    description         TEXT NOT NULL,
    source              TEXT NOT NULL,  -- GHG Protocol | ESRS E1
    tier_level          INTEGER NOT NULL,  -- 1 | 2 | 3
    priority            TEXT NOT NULL,     -- high | medium | low
    business_term_id    INTEGER,
    technical_field_id  INTEGER,
    coverage_status     TEXT DEFAULT 'offen',  -- offen | partiell | vollständig
    FOREIGN KEY (business_term_id)  REFERENCES business_terms(id),
    FOREIGN KEY (technical_field_id) REFERENCES technical_fields(id)
)
""")

# 6. Metadaten-Lücken (wird durch Gap Detection befüllt)
cursor.execute("""
CREATE TABLE IF NOT EXISTS metadata_gaps (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    gap_type            TEXT NOT NULL,
    severity            TEXT NOT NULL,  -- high | medium | low
    affected_entity     TEXT NOT NULL,  -- Tabellenname
    affected_id         INTEGER NOT NULL,
    description         TEXT NOT NULL,
    recommended_action  TEXT,
    status              TEXT DEFAULT 'offen',  -- offen | in_bearbeitung | geschlossen
    created_at          TEXT DEFAULT (datetime('now'))
)
""")

conn.commit()
print("✓ Schema erfolgreich erstellt — 6 Tabellen angelegt")
print("✓ Fremdschlüssel aktiv")
print("✓ catalog.db bereit für Seed-Daten")

✓ Schema erfolgreich erstellt — 6 Tabellen angelegt
✓ Fremdschlüssel aktiv
✓ catalog.db bereit für Seed-Daten


In [4]:
# Prüfzelle: Tabellen anzeigen
tables = pd.read_sql_query("""
    SELECT name FROM sqlite_master 
    WHERE type='table' 
    ORDER BY name
""", conn)

print("Angelegte Tabellen:")
print(tables.to_string(index=False))

Angelegte Tabellen:
            name
  business_terms
        datasets
   metadata_gaps
    requirements
  source_systems
 sqlite_sequence
technical_fields


In [15]:
# Prüfzelle: Überblick Seed-Daten
for table in ["source_systems", "datasets", "technical_fields", 
              "business_terms", "requirements"]:
    count = pd.read_sql_query(
        f"SELECT COUNT(*) as anzahl FROM {table}", conn
    )
    print(f"{table:25s} → {count['anzahl'][0]} Einträge")

source_systems            → 9 Einträge
datasets                  → 10 Einträge
technical_fields          → 18 Einträge
business_terms            → 10 Einträge
requirements              → 11 Einträge
